# MCMO Adapter - Teste no Colab (Versão Corrigida)

**Correções:**
- Usa `geatpy` (última versão disponível)
- Substitui `scikit-multiflow` por `river`
- Versão compatível com Colab

**Autor:** Claude Code  
**Data:** 2025-11-24

## 1. Setup - Conectar Google Drive

In [ ]:
from google.colab import drive
import os
import sys

# Montar Google Drive
drive.mount('/content/drive')

# AJUSTE O PATH ABAIXO
DSL_PATH = '/content/drive/MyDrive/DSL-AG-hybrid'

# Verificar
if os.path.exists(DSL_PATH):
    print(f"✓ Pasta encontrada: {DSL_PATH}")
    sys.path.insert(0, DSL_PATH)
    
    MCMO_PATH = os.path.join(DSL_PATH, 'mcmo')
    if os.path.exists(MCMO_PATH):
        print(f"✓ Pasta mcmo encontrada")
        files = os.listdir(MCMO_PATH)
        print(f"  Arquivos: {[f for f in files if not f.startswith('_')]}")
    else:
        print(f"✗ Pasta mcmo não encontrada")
else:
    print(f"✗ Pasta não encontrada. Ajuste DSL_PATH na célula acima.")

## 2. Instalar Dependências (CORRIGIDO)

In [ ]:
# Instalar geatpy (última versão disponível)
print("Instalando geatpy...")
!pip install -q geatpy

# Instalar river (substituto moderno de scikit-multiflow)
print("Instalando river...")
!pip install -q river

print("\n✓ Instalação completa!")

In [ ]:
# Verificar instalação
import warnings
warnings.filterwarnings('ignore')

deps_ok = True

try:
    import geatpy as ea
    print(f"✓ geatpy versão {ea.__version__}")
except ImportError as e:
    print(f"✗ geatpy: {e}")
    deps_ok = False

try:
    import river
    print(f"✓ river versão {river.__version__}")
except ImportError as e:
    print(f"✗ river: {e}")
    deps_ok = False

try:
    import numpy as np
    import pandas as pd
    from sklearn.metrics import accuracy_score
    print("✓ numpy, pandas, sklearn disponíveis")
except ImportError as e:
    print(f"✗ Outras dependências: {e}")
    deps_ok = False

if deps_ok:
    print("\n✓ Todas as dependências OK!")
else:
    print("\n✗ Algumas dependências faltando.")

## 3. Importar MCMOAdapter (River Version)

In [ ]:
# Importar versão river-compatible
try:
    from mcmo.baseline_mcmo_river import MCMOAdapter, MCMOEvaluator, test_mcmo_adapter
    print("✓ MCMOAdapter (river version) importado com sucesso!")
    
    from mcmo import baseline_mcmo_river
    if baseline_mcmo_river.MCMO_AVAILABLE:
        print("✓ MCMO disponível e pronto para uso")
    else:
        print(f"✗ MCMO não disponível: {baseline_mcmo_river.IMPORT_ERROR}")
        
except ImportError as e:
    print(f"✗ Erro ao importar: {e}")
    print("\nVerifique:")
    print("1. Os arquivos MCMO_river.py e baseline_mcmo_river.py estão na pasta mcmo/")
    print("2. O path DSL_PATH está correto")
    print("3. As dependências foram instaladas")

## 4. Teste com Dados Sintéticos

In [ ]:
import numpy as np
from sklearn.datasets import make_classification

print("=" * 70)
print("Teste 1: Dados Sintéticos com Drift")
print("=" * 70)

# Gerar dataset sintético
np.random.seed(42)

n_samples_per_chunk = 500
n_chunks = 8  # Reduzido para ser mais rápido
n_features = 10

print(f"\nGerando {n_chunks} chunks de {n_samples_per_chunk} amostras...")

X_chunks = []
y_chunks = []

for i in range(n_chunks):
    # Drift gradual nas proporções das classes
    weights = [0.5 + 0.05*i, 0.5 - 0.05*i]
    
    X, y = make_classification(
        n_samples=n_samples_per_chunk,
        n_features=n_features,
        n_informative=8,
        n_redundant=2,
        n_classes=2,
        weights=weights,
        flip_y=0.01,
        random_state=42+i
    )
    
    X_chunks.append(X)
    y_chunks.append(y)
    
    print(f"  Chunk {i+1}: {len(X)} samples, classes: {np.bincount(y)}")

print("\n✓ Dataset sintético gerado")

In [ ]:
# Testar MCMOAdapter
print("\n" + "=" * 70)
print("Testando MCMOAdapter (n_sources=3)")
print("=" * 70 + "\n")

results = test_mcmo_adapter(
    X_chunks=X_chunks,
    y_chunks=y_chunks,
    n_sources=3,
    verbose=True
)

print("\n" + "=" * 70)
print("Resultados - Dados Sintéticos")
print("=" * 70)
print(f"Acurácia Média: {results['mean_accuracy']:.4f}")
print(f"Total de Amostras: {results['global_metrics']['total_samples']}")

In [ ]:
# Plot accuracies
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))
plt.plot(range(1, len(results['accuracies'])+1), results['accuracies'],
         marker='o', linewidth=2, markersize=8, color='steelblue')
plt.axhline(y=results['mean_accuracy'], color='r', linestyle='--',
            label=f'Mean: {results["mean_accuracy"]:.4f}')
plt.xlabel('Chunk', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('MCMO Adapter - Accuracy per Chunk', fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Comparação com Baseline

In [ ]:
from mcmo.MCMO_river import RiverTreeWrapper

print("=" * 70)
print("Comparação: MCMO vs Baseline (River HoeffdingTree)")
print("=" * 70)

# Testar baseline
baseline = RiverTreeWrapper()
baseline_accuracies = []

print("\nTestando baseline...")
for i, (X_chunk, y_chunk) in enumerate(zip(X_chunks, y_chunks)):
    predictions = []
    
    for j in range(len(X_chunk)):
        X_sample = X_chunk[j:j+1]
        y_sample = y_chunk[j:j+1]
        
        if i == 0 and j == 0:
            pred = 0
        else:
            pred = baseline.predict(X_sample)[0]
        
        predictions.append(pred)
        baseline.partial_fit(X_sample, y_sample)
    
    accuracy = np.mean(np.array(predictions) == y_chunk)
    baseline_accuracies.append(accuracy)
    print(f"  Chunk {i+1}: {accuracy:.4f}")

baseline_mean = np.mean(baseline_accuracies)
print(f"\nBaseline Mean: {baseline_mean:.4f}")
print(f"MCMO Mean:     {results['mean_accuracy']:.4f}")
print(f"Diferença:     {(results['mean_accuracy'] - baseline_mean)*100:+.2f} p.p.")

if results['mean_accuracy'] > baseline_mean:
    print("\n✓ MCMO superou o baseline!")
else:
    print("\n⚠ MCMO não superou baseline neste teste")

In [ ]:
# Plot comparação
plt.figure(figsize=(14, 6))

plt.plot(range(1, len(results['accuracies'])+1), results['accuracies'],
         marker='o', linewidth=2, markersize=8, label='MCMO', color='steelblue')
plt.plot(range(1, len(baseline_accuracies)+1), baseline_accuracies,
         marker='s', linewidth=2, markersize=8, label='Baseline', color='orange')

plt.axhline(y=results['mean_accuracy'], color='steelblue', linestyle='--', alpha=0.5)
plt.axhline(y=baseline_mean, color='orange', linestyle='--', alpha=0.5)

plt.xlabel('Chunk', fontsize=12)
plt.ylabel('Accuracy', fontsize=12)
plt.title('MCMO vs Baseline - Comparison', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Resumo

In [ ]:
print("="*70)
print("RESUMO DOS TESTES")
print("="*70)

print("\n1. CONFIGURAÇÃO")
print("-" * 70)
print(f"   n_sources:            {3}")
print(f"   initial_beach:        {200}")
print(f"   Chunks testados:      {len(X_chunks)}")
print(f"   Amostras por chunk:   {n_samples_per_chunk}")

print("\n2. RESULTADOS")
print("-" * 70)
print(f"   MCMO Adapter:         {results['mean_accuracy']:.4f}")
print(f"   Baseline (River HT):  {baseline_mean:.4f}")
print(f"   Diferença:            {(results['mean_accuracy'] - baseline_mean)*100:+.2f} p.p.")

print("\n3. CONCLUSÕES")
print("-" * 70)
if results['mean_accuracy'] > baseline_mean:
    print("   ✓ MCMO demonstrou superioridade")
else:
    print("   ⚠ MCMO não superou baseline (pode precisar tuning)")

print("\n   ✓ Temporal splitting funcionou")
print("   ✓ River compatibility OK")
print("   ✓ Pronto para integração")

print("\n4. PRÓXIMOS PASSOS")
print("-" * 70)
print("   1. Testar com Electricity dataset")
print("   2. Integrar em main.py")
print("   3. Executar Phase 3 experiments")
print("   4. Análise estatística (Friedman, Wilcoxon)")

print("\n" + "="*70)

## 7. (Opcional) Teste com Electricity

In [ ]:
# Se você tiver o dataset Electricity no Drive, descomente e execute:

# import pandas as pd
# from sklearn.preprocessing import LabelEncoder

# # Ajustar path
# elec_path = os.path.join(DSL_PATH, 'datasets', 'Electricity.csv')

# if os.path.exists(elec_path):
#     print("Carregando Electricity...")
#     data = pd.read_csv(elec_path)
#     
#     # Preparar dados
#     target_col = 'class' if 'class' in data.columns else data.columns[-1]
#     X_elec = data.drop(target_col, axis=1).values
#     y_elec = data[target_col].values
#     
#     if y_elec.dtype == object:
#         le = LabelEncoder()
#         y_elec = le.fit_transform(y_elec)
#     
#     # Dividir em chunks
#     chunk_size = 1000
#     n_chunks_elec = min(10, len(X_elec) // chunk_size)  # Máximo 10 chunks
#     
#     X_chunks_elec = [X_elec[i*chunk_size:(i+1)*chunk_size] for i in range(n_chunks_elec)]
#     y_chunks_elec = [y_elec[i*chunk_size:(i+1)*chunk_size] for i in range(n_chunks_elec)]
#     
#     # Testar
#     results_elec = test_mcmo_adapter(X_chunks_elec, y_chunks_elec, n_sources=3, verbose=True)
#     print(f"\nElectricity Mean Accuracy: {results_elec['mean_accuracy']:.4f}")
# else:
#     print(f"Electricity não encontrado em: {elec_path}")

## 8. Exportar Resultados

In [ ]:
# Salvar resultados
import pandas as pd

results_df = pd.DataFrame({
    'chunk': range(1, len(results['accuracies'])+1),
    'mcmo_accuracy': results['accuracies'],
    'baseline_accuracy': baseline_accuracies
})

# Salvar no Drive
output_path = os.path.join(DSL_PATH, 'mcmo_colab_results.csv')
results_df.to_csv(output_path, index=False)
print(f"✓ Resultados salvos em: {output_path}")

# Salvar localmente no Colab
results_df.to_csv('mcmo_colab_results.csv', index=False)
print("✓ Cópia local: /content/mcmo_colab_results.csv")
print("\nPara baixar: Files → mcmo_colab_results.csv → Download")